In [2]:
import sqlite3
import pandas as pd

# Load dataset
df = pd.read_csv(r"C:\Ujwal\RIL\Data set\transportation-cost-driver-analysis\data\superstore.csv")

# Create SQLite in-memory database
conn = sqlite3.connect(":memory:")

# Push dataframe to SQL
df.to_sql("orders", conn, index=False, if_exists="replace")

# Quick check
pd.read_sql("SELECT COUNT(*) as total_rows FROM orders", conn)

,total_rows
0,51290


In [3]:
pd.read_sql("PRAGMA table_info(orders);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,Category,TEXT,0,None,0
1,1,City,TEXT,0,None,0
2,2,Country,TEXT,0,None,0
3,3,Customer.ID,TEXT,0,None,0
4,4,Customer.Name,TEXT,0,None,0
5,5,Discount,REAL,0,None,0
6,6,Market,TEXT,0,None,0
7,7,记录数,INTEGER,0,None,0
8,8,Order.Date,TEXT,0,None,0
9,9,Order.ID,TEXT,0,None,0


In [4]:
pd.read_sql("SELECT * FROM orders LIMIT 5;", conn)

,Category,City,Country,Customer.ID,Customer.Name,Discount,Market,记录数,Order.Date,Order.ID,...,Sales,Segment,Ship.Date,Ship.Mode,Shipping.Cost,State,Sub.Category,Year,Market2,weeknum
0,Office Supplies,Los Angeles,United States,LS-172304,Lycoris Saunders,0.0,US,1,2011-01-07 00:00:00.000,CA-2011-130813,...,19,Consumer,2011-01-09 00:00:00.000,Second Class,4.37,California,Paper,2011,North America,2
1,Office Supplies,Los Angeles,United States,MV-174854,Mark Van Huff,0.0,US,1,2011-01-21 00:00:00.000,CA-2011-148614,...,19,Consumer,2011-01-26 00:00:00.000,Standard Class,0.94,California,Paper,2011,North America,4
2,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,1,2011-08-05 00:00:00.000,CA-2011-118962,...,21,Consumer,2011-08-09 00:00:00.000,Standard Class,1.81,California,Paper,2011,North America,32
3,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,1,2011-08-05 00:00:00.000,CA-2011-118962,...,111,Consumer,2011-08-09 00:00:00.000,Standard Class,4.59,California,Paper,2011,North America,32
4,Office Supplies,Los Angeles,United States,AP-109154,Arthur Prichep,0.0,US,1,2011-09-29 00:00:00.000,CA-2011-146969,...,6,Consumer,2011-10-03 00:00:00.000,Standard Class,1.32,California,Paper,2011,North America,40


In [5]:
df.columns

Index(['Category', 'City', 'Country', 'Customer.ID', 'Customer.Name',
       'Discount', 'Market', '记录数', 'Order.Date', 'Order.ID', 'Order.Priority',
       'Product.ID', 'Product.Name', 'Profit', 'Quantity', 'Region', 'Row.ID',
       'Sales', 'Segment', 'Ship.Date', 'Ship.Mode', 'Shipping.Cost', 'State',
       'Sub.Category', 'Year', 'Market2', 'weeknum'],
      dtype='object')

In [7]:
df.columns = df.columns.str.replace(".", "_", regex=False)
df.to_sql("orders", conn, index=False, if_exists="replace")

51290

In [8]:
pd.read_sql("""
SELECT 
    Order_ID,
    COUNT(*) as line_items,
    SUM(Shipping_Cost) as total_shipping,
    MIN(Shipping_Cost) as min_shipping,
    MAX(Shipping_Cost) as max_shipping
FROM orders
GROUP BY Order_ID
LIMIT 10;
""", conn)

,Order_ID,line_items,total_shipping,min_shipping,max_shipping
0,AE-2011-9160,2,9.56,3.87,5.69
1,AE-2013-1130,2,60.18,0.10,60.08
2,AE-2013-1530,2,3.16,1.41,1.75
3,AE-2014-2840,1,8.04,8.04,8.04
4,AE-2014-3830,6,19.38,0.25,6.73
5,AE-2014-4120,1,0.49,0.49,0.49
6,AG-2011-1070,1,31.41,31.41,31.41
7,AG-2011-1390,2,37.73,2.41,35.32
8,AG-2011-1440,1,3.04,3.04,3.04
9,AG-2011-2040,1,35.46,35.46,35.46


In [9]:
order_level = pd.read_sql("""
SELECT
    Order_ID,
    MIN(Order_Date) as Order_Date,
    MIN(Region) as Region,
    MIN(Ship_Mode) as Ship_Mode,
    SUM(Sales) as Total_Sales,
    SUM(Profit) as Total_Profit,
    SUM(Shipping_Cost) as Total_Shipping_Cost,
    SUM(Quantity) as Total_Quantity
FROM orders
GROUP BY Order_ID
""", conn)

order_level.head()

,Order_ID,Order_Date,Region,Ship_Mode,Total_Sales,Total_Profit,Total_Shipping_Cost,Total_Quantity
0,AE-2011-9160,2011-10-03 00:00:00.000,EMEA,Standard Class,161,-246.078,9.56,8
1,AE-2013-1130,2013-10-14 00:00:00.000,EMEA,Same Day,229,-236.964,60.18,7
2,AE-2013-1530,2013-12-31 00:00:00.000,EMEA,Second Class,24,-38.076,3.16,3
3,AE-2014-2840,2014-11-05 00:00:00.000,EMEA,First Class,42,-75.060,8.04,1
4,AE-2014-3830,2014-12-13 00:00:00.000,EMEA,Standard Class,280,-429.108,19.38,16


In [10]:
order_level["Shipping_Ratio"] = order_level["Total_Shipping_Cost"] / order_level["Total_Sales"]
order_level["Profit_Margin"] = order_level["Total_Profit"] / order_level["Total_Sales"]
order_level["Shipping_per_Unit"] = order_level["Total_Shipping_Cost"] / order_level["Total_Quantity"]

order_level.head()

,Order_ID,Order_Date,Region,Ship_Mode,Total_Sales,Total_Profit,Total_Shipping_Cost,Total_Quantity,Shipping_Ratio,Profit_Margin,Shipping_per_Unit
0,AE-2011-9160,2011-10-03 00:00:00.000,EMEA,Standard Class,161,-246.078,9.56,8,0.059379,-1.528435,1.195000
1,AE-2013-1130,2013-10-14 00:00:00.000,EMEA,Same Day,229,-236.964,60.18,7,0.262795,-1.034777,8.597143
2,AE-2013-1530,2013-12-31 00:00:00.000,EMEA,Second Class,24,-38.076,3.16,3,0.131667,-1.586500,1.053333
3,AE-2014-2840,2014-11-05 00:00:00.000,EMEA,First Class,42,-75.060,8.04,1,0.191429,-1.787143,8.040000
4,AE-2014-3830,2014-12-13 00:00:00.000,EMEA,Standard Class,280,-429.108,19.38,16,0.069214,-1.532529,1.211250


In [11]:
order_level["Order_Date"] = pd.to_datetime(order_level["Order_Date"])
order_level["Year_Month"] = order_level["Order_Date"].dt.to_period("M")

In [12]:
monthly = order_level.groupby("Year_Month").agg(
    Total_Sales=("Total_Sales", "sum"),
    Total_Shipping=("Total_Shipping_Cost", "sum"),
    Total_Profit=("Total_Profit", "sum")
).reset_index()

monthly["Shipping_Ratio"] = monthly["Total_Shipping"] / monthly["Total_Sales"]
monthly["Profit_Margin"] = monthly["Total_Profit"] / monthly["Total_Sales"]

monthly.head()

,Year_Month,Total_Sales,Total_Shipping,Total_Profit,Shipping_Ratio,Profit_Margin
0,2011-01,102561,10820.3280,8558.78446,0.105501,0.083451
1,2011-02,95696,11054.3330,11745.57548,0.115515,0.122738
2,2011-03,147910,13232.7755,15825.45876,0.089465,0.106994
3,2011-04,120094,13377.1500,13127.52558,0.111389,0.109310
4,2011-05,151106,17062.1760,12718.00650,0.112915,0.084166


In [13]:
monthly[["Shipping_Ratio", "Profit_Margin"]].describe()

,Shipping_Ratio,Profit_Margin
count,48.000000,48.000000
mean,0.106910,0.114580
std,0.005371,0.018889
min,0.089465,0.057933
25%,0.103122,0.103011
50%,0.106985,0.117403
75%,0.110860,0.127386
max,0.120312,0.147785


In [15]:
monthly.sort_values("Year_Month").tail()

,Year_Month,Total_Sales,Total_Shipping,Total_Profit,Shipping_Ratio,Profit_Margin
43,2014-08,450351,45694.203,53216.37976,0.101464,0.118166
44,2014-09,468159,51605.555,59483.59990,0.110231,0.127059
45,2014-10,406857,42697.376,55202.51826,0.104944,0.135680
46,2014-11,533055,57533.840,59598.47918,0.107932,0.111805
47,2014-12,460318,51156.747,45042.14418,0.111133,0.097850


In [16]:
order_level["High_Shipping"] = (order_level["Shipping_Ratio"] > order_level["Shipping_Ratio"].median()).astype(int)

order_level.groupby("High_Shipping")[["Shipping_Ratio", "Profit_Margin"]].mean()

,Shipping_Ratio,Profit_Margin
High_Shipping,,
0,0.062620,0.045549
1,0.156222,0.049132


In [17]:
order_level[["Shipping_Ratio", "Profit_Margin", "Total_Sales", "Total_Quantity"]].corr()

,Shipping_Ratio,Profit_Margin,Total_Sales,Total_Quantity
Shipping_Ratio,1.000000,0.000955,-0.022681,-0.014903
Profit_Margin,0.000955,1.000000,0.097387,0.024457
Total_Sales,-0.022681,0.097387,1.000000,0.539715
Total_Quantity,-0.014903,0.024457,0.539715,1.000000


In [18]:
discount_analysis = pd.read_sql("""
SELECT
    Order_ID,
    SUM(Discount) as Total_Discount
FROM orders
GROUP BY Order_ID
""", conn)

order_level = order_level.merge(discount_analysis, on="Order_ID")

order_level["Discount_Ratio"] = order_level["Total_Discount"]

order_level[["Discount_Ratio", "Profit_Margin"]].corr()

,Discount_Ratio,Profit_Margin
Discount_Ratio,1.000000,-0.635525
Profit_Margin,-0.635525,1.000000


In [19]:
import statsmodels.api as sm

X = order_level[["Shipping_Ratio", "Discount_Ratio", "Total_Sales", "Total_Quantity"]]
X = sm.add_constant(X)
y = order_level["Profit_Margin"]

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:          Profit_Margin   R-squared:                       0.454
Model:                            OLS   Adj. R-squared:                  0.454
Method:                 Least Squares   F-statistic:                     5197.
Date:                Thu, 26 Feb 2026   Prob (F-statistic):               0.00
Time:                        18:42:01   Log-Likelihood:                -7493.0
No. Observations:               25035   AIC:                         1.500e+04
Df Residuals:                   25030   BIC:                         1.504e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.0927      0.005     19.

In [20]:
avg_discount = order_level["Discount_Ratio"].mean()
avg_margin = order_level["Profit_Margin"].mean()

impact_5pct = -0.5668 * 0.05

avg_margin, impact_5pct

(0.04734018690254952, -0.02834)

In [21]:
total_revenue = order_level["Total_Sales"].sum()

margin_loss_pct = 0.02834  # from previous step
annual_margin_loss = total_revenue * margin_loss_pct

total_revenue, annual_margin_loss

(12642905, 358299.9277)

In [22]:
segment_analysis = pd.read_sql("""
SELECT
    Segment,
    SUM(Sales) as Total_Sales,
    SUM(Profit) as Total_Profit,
    SUM(Discount) as Total_Discount
FROM orders
GROUP BY Segment
""", conn)

segment_analysis["Profit_Margin"] = segment_analysis["Total_Profit"] / segment_analysis["Total_Sales"]
segment_analysis["Discount_Ratio"] = segment_analysis["Total_Discount"]

segment_analysis

,Segment,Total_Sales,Total_Profit,Total_Discount,Profit_Margin,Discount_Ratio
0,Consumer,6508141,749239.78206,3808.042,0.115123,3808.042
1,Corporate,3824808,441208.32866,2205.284,0.115354,2205.284
2,Home Office,2309956,277009.18056,1316.402,0.119920,1316.402


In [23]:
segment_analysis["Discount_Ratio"] = segment_analysis["Total_Discount"] / segment_analysis["Total_Sales"]

segment_analysis

,Segment,Total_Sales,Total_Profit,Total_Discount,Profit_Margin,Discount_Ratio
0,Consumer,6508141,749239.78206,3808.042,0.115123,0.000585
1,Corporate,3824808,441208.32866,2205.284,0.115354,0.000577
2,Home Office,2309956,277009.18056,1316.402,0.119920,0.000570


In [24]:
discount_corrected = pd.read_sql("""
SELECT
    Segment,
    SUM(Sales) as Total_Sales,
    SUM(Profit) as Total_Profit,
    SUM(Sales * Discount) as Discount_Value
FROM orders
GROUP BY Segment
""", conn)

discount_corrected["Profit_Margin"] = discount_corrected["Total_Profit"] / discount_corrected["Total_Sales"]
discount_corrected["Discount_Ratio"] = discount_corrected["Discount_Value"] / discount_corrected["Total_Sales"]

discount_corrected

,Segment,Total_Sales,Total_Profit,Discount_Value,Profit_Margin,Discount_Ratio
0,Consumer,6508141,749239.78206,691925.524,0.115123,0.106317
1,Corporate,3824808,441208.32866,404434.178,0.115354,0.105740
2,Home Office,2309956,277009.18056,250015.582,0.119920,0.108234


In [26]:
import numpy as np

order_level["Discount_Bucket"] = pd.cut(
    order_level["Discount_Ratio"],
    bins=[-0.001, 0.0001, 0.10, 0.20, 1],
    labels=["No Discount", "Low (0-10%)", "Medium (10-20%)", "High (>20%)"]
)

bucket_analysis = order_level.groupby("Discount_Bucket").agg(
    Avg_Margin=("Profit_Margin", "mean"),
    Avg_Discount=("Discount_Ratio", "mean"),
    Order_Count=("Order_ID", "count")
).reset_index()

bucket_analysis

,Discount_Bucket,Avg_Margin,Avg_Discount,Order_Count
0,No Discount,0.260901,0.000000,12883
1,Low (0-10%),0.188455,0.079056,1786
2,Medium (10-20%),0.155518,0.193873,2685
3,High (>20%),-0.290731,0.545444,5698


In [27]:
high_discount_analysis = pd.read_sql("""
SELECT
    Category,
    COUNT(*) as Line_Items,
    SUM(Sales) as Total_Sales,
    SUM(Profit) as Total_Profit,
    SUM(Sales * Discount) / SUM(Sales) as Discount_Ratio
FROM orders
WHERE Discount > 0.20
GROUP BY Category
""", conn)

high_discount_analysis["Profit_Margin"] = high_discount_analysis["Total_Profit"] / high_discount_analysis["Total_Sales"]

high_discount_analysis

,Category,Line_Items,Total_Sales,Total_Profit,Discount_Ratio,Profit_Margin
0,Furniture,2859,904321,-319762.7365,0.405150,-0.353594
1,Office Supplies,6381,416969,-237758.6512,0.503708,-0.570207
2,Technology,2088,608952,-257160.6990,0.450325,-0.422300


In [28]:
no_discount_analysis = pd.read_sql("""
SELECT
    Category,
    COUNT(*) as Line_Items,
    SUM(Sales) as Total_Sales,
    SUM(Profit) as Total_Profit
FROM orders
WHERE Discount = 0
GROUP BY Category
""", conn)

no_discount_analysis["Profit_Margin"] = (
    no_discount_analysis["Total_Profit"] /
    no_discount_analysis["Total_Sales"]
)

no_discount_analysis

,Category,Line_Items,Total_Sales,Total_Profit,Profit_Margin
0,Furniture,4496,1925169,465612.9564,0.241856
1,Office Supplies,19209,2378027,608725.2864,0.255979
2,Technology,5304,2689538,696357.0304,0.258913


In [29]:
high_discount_revenue = pd.read_sql("""
SELECT
    SUM(Sales) as High_Discount_Sales
FROM orders
WHERE Discount > 0.20
""", conn)

total_revenue = pd.read_sql("""
SELECT
    SUM(Sales) as Total_Sales
FROM orders
""", conn)

high_discount_revenue, total_revenue

(   High_Discount_Sales
 0              1930242,
    Total_Sales
 0     12642905)

In [30]:
share = high_discount_revenue.iloc[0,0] / total_revenue.iloc[0,0]
share

0.15267393055630807

In [31]:
loss_from_high_discount = pd.read_sql("""
SELECT
    SUM(Profit) as Total_Profit
FROM orders
WHERE Discount > 0.20
""", conn)

loss_from_high_discount

,Total_Profit
0,-814682.0867


In [32]:
total_profit = pd.read_sql("""
SELECT SUM(Profit) as Total_Profit
FROM orders
""", conn)

total_profit

,Total_Profit
0,1.467457e+06


In [36]:
order_level[order_level["Discount_Ratio"] >= 1]

,Order_ID,Order_Date,Region,Ship_Mode,Total_Sales,Total_Profit,Total_Shipping_Cost,Total_Quantity,Shipping_Ratio,Profit_Margin,Shipping_per_Unit,Year_Month,High_Shipping,Total_Discount,Discount_Ratio,Discount_Bucket,Adjusted_Profit
0,AE-2011-9160,2011-10-03,EMEA,Standard Class,161,-246.0780,9.56,8,0.059379,-1.528435,1.195000,2011-10,0,1.4,1.4,NaN,-439.2780
1,AE-2013-1130,2013-10-14,EMEA,Same Day,229,-236.9640,60.18,7,0.262795,-1.034777,8.597143,2013-10,1,1.4,1.4,NaN,-511.7640
2,AE-2013-1530,2013-12-31,EMEA,Second Class,24,-38.0760,3.16,3,0.131667,-1.586500,1.053333,2013-12,1,1.4,1.4,NaN,-66.8760
4,AE-2014-3830,2014-12-13,EMEA,Standard Class,280,-429.1080,19.38,16,0.069214,-1.532529,1.211250,2014-12,0,4.2,4.2,NaN,-1549.1080
378,CA-2011-102869,2011-09-09,East,Second Class,234,-26.0068,14.56,13,0.062222,-0.111140,1.120000,2011-09,0,1.0,1.0,High (>20%),-213.2068
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25026,ZI-2014-5970,2014-06-10,Africa,Standard Class,719,-1165.0500,87.40,19,0.121558,-1.620376,4.600000,2014-06,1,2.1,2.1,NaN,-2531.1500
25028,ZI-2014-6740,2014-11-30,Africa,Standard Class,245,-225.0090,39.91,14,0.162898,-0.918404,2.850714,2014-11,1,4.2,4.2,NaN,-1205.0090
25030,ZI-2014-7160,2014-10-30,Africa,First Class,52,-96.1350,3.72,4,0.071538,-1.848750,0.930000,2014-10,0,2.1,2.1,NaN,-194.9350
25031,ZI-2014-7610,2014-03-24,Africa,Standard Class,26,-30.3120,2.34,2,0.090000,-1.165846,1.170000,2014-03,0,1.4,1.4,NaN,-61.5120


In [37]:
def adjusted_profit(row):
    if row["Discount_Ratio"] > 0.20 and row["Discount_Ratio"] < 1:
        original_sales = row["Total_Sales"]
        original_discount = row["Discount_Ratio"]
        cost = original_sales - row["Total_Profit"]

        list_price = original_sales / (1 - original_discount)
        new_sales = list_price * (1 - 0.20)

        return new_sales - cost
    else:
        return row["Total_Profit"]

order_level["Adjusted_Profit"] = order_level.apply(adjusted_profit, axis=1)

adjusted_profit_total = order_level["Adjusted_Profit"].sum()
adjusted_profit_total

4912271.07771303

In [39]:
adjusted_profit_total - total_profit.iloc[0,0]

3444813.786433044

In [40]:
retention_rates = [1.0, 0.75, 0.5, 0.25]

results = []

for r in retention_rates:
    simulated_profit = total_profit.iloc[0,0] + r * (adjusted_profit_total - total_profit.iloc[0,0])
    results.append((r, simulated_profit))

results

[(1.0, 4912271.07771303),
 (0.75, 4051067.631104769),
 (0.5, 3189864.184496509),
 (0.25, 2328660.7378882477)]

In [41]:
order_level.to_csv("order_level.csv", index=False)
monthly.to_csv("monthly_summary.csv", index=False)
high_discount_analysis.to_csv("high_discount_category.csv", index=False)
bucket_analysis.to_csv("discount_bucket_analysis.csv", index=False)

In [42]:
import pandas as pd

scenario_df = pd.DataFrame(results, columns=["Retention_Rate", "Simulated_Profit"])
scenario_df.to_csv("scenario_sensitivity.csv", index=False)